# ADP1 BERDL Cross-Sample Consistency Analysis

## Objectives

This notebook performs cross-sample consistency analysis across all 6 mutant strains,
building on the fold-change flux analysis from ADP1BERDLFoldChangeAnalysis.

### Analysis Goals:
1. Load fold change FBA results and gene-level fold changes for all 6 strains
2. Compute three data channels per reaction per strain: proteomics FC, flux FC, fitted flux
3. Identify reactions consistently changed across multiple strains
4. Classify agreement/conflict between proteomics and flux changes
5. Generate enhanced Escher maps with reaction badges showing proteomics and flux FC
6. Create an HTML index page linking all maps

### Strains (Target Conditions):
- **ACN2586**: Initial construct
- **ACN2821**: Evolved strain
- **ACN3425, ACN3427, ACN3429, ACN3430**: Additional strain variants

### Reference Condition:
- **ADP1**: Wild-type Acinetobacter baylyi ADP1

### Reference Flux:
- **BERDL fitness-constrained flux** on pyruvate media (from ADP1BERDLFitnessFluxFitting)

## Section A: Load All Results

Load from datacache:
- Fold change FBA results for all 6 strains
- Reference flux distribution
- Gene-level fold change files for each strain

In [1]:
%run util.py

# Target strains
target_strains = ["ACN2586", "ACN2821", "ACN3425", "ACN3427", "ACN3429", "ACN3430"]
full_condition_names = {s: f"Pyruvate_{s}_DgoA_2025" for s in target_strains}
reference_condition = "Pyruvate_ADP1_DgoA_2025"

# Load fold change FBA results (all 6 strains)
fc_results = util.load("ADP1BERDLFoldChangeAnalysis/ADP1BERDLFoldChangeAnalysis")
print(f"Loaded fold change FBA results: {len(fc_results)} conditions")
print(f"  Conditions: {list(fc_results.keys())}")

# Load reference flux
reference_flux = util.load("ADP1BERDLFoldChangeAnalysis/berdl_fc_reference_flux")
print(f"\nLoaded reference flux: {len(reference_flux)} reactions")
print(f"  Non-zero: {sum(1 for v in reference_flux.values() if abs(v) > 1e-9)}")

# Load gene-level fold changes for each strain
gene_fold_changes = {}
for strain in target_strains:
    cache_key = f"berdl_fc_{strain}_vs_ADP1"
    gene_fc = util.load("ADP1BERDLFoldChangeAnalysis/"+cache_key)
    gene_fold_changes[strain] = gene_fc
    print(f"  {strain}: {len(gene_fc)} genes with fold change data")

# Load model for gene-reaction rules
model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")
print(f"\nModel loaded: {len(model.model.reactions)} reactions, {len(model.model.genes)} genes")

# Save loaded data references for subsequent cells
util.save("cross_sample_strains", {
    "target_strains": target_strains,
    "full_condition_names": full_condition_names,
    "reference_condition": reference_condition
})

print("\nAll data loaded successfully.")

/Users/chenry/Dropbox/Projects/KBUtilLib/src


[KBUtilLib] Failed to import rcsb_pdb_utils: ModuleNotFoundError: No module named 'aiohttp'


modelseedpy 0.4.2


2026-03-16 13:07:54,244 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-03-16 13:07:54,245 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-03-16 13:07:54,246 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token


loading biochemistry database from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


2026-03-16 13:07:59,202 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase
2026-03-16 13:07:59,635 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-03-16 13:07:59,637 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLCrossSampleAnalysis
2026-03-16 13:07:59,638 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-03-16 13:07:59,639 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


cobrakbase 0.4.0
/Users/chenry/.npm-global/bin/claude


2026-03-16 13:07:59,899 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code
2026-03-16 13:07:59,900 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1BERDLCrossSampleAnalysis, loading from base datacache
2026-03-16 13:07:59,938 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1BERDLCrossSampleAnalysis, loading from base datacache
2026-03-16 13:07:59,939 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1BERDLCrossSampleAnalysis, loading from base datacache
2026-03-16 13:07:59,941 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1BERDLCrossSampleAnalysis, loading from base datacache
2026-03-16 13:07:59,942 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/

Loaded fold change FBA results: 6 conditions
  Conditions: ['Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025']

Loaded reference flux: 1504 reactions
  Non-zero: 421
  ACN2586: 2383 genes with fold change data
  ACN2821: 2383 genes with fold change data
  ACN3425: 2383 genes with fold change data
  ACN3427: 2383 genes with fold change data
  ACN3429: 2383 genes with fold change data
  ACN3430: 2383 genes with fold change data

Model loaded: 1504 reactions, 1840 genes

All data loaded successfully.


## Section B: Compute Three Data Channels Per Reaction Per Strain

For each reaction in each strain, compute:
1. **Proteomics fold change**: Gene-level FC mapped to reactions (geometric mean for multi-gene reactions)
2. **Flux fold change**: abs(fitted_flux) / abs(reference_flux)
3. **Fitted flux value**

In [2]:
%run util.py
import numpy as np
import re

# Load all required data
strain_info = util.load("cross_sample_strains")
target_strains = strain_info["target_strains"]
full_condition_names = strain_info["full_condition_names"]

fc_results = util.load("ADP1BERDLFoldChangeAnalysis/ADP1BERDLFoldChangeAnalysis")
reference_flux = util.load("ADP1BERDLFoldChangeAnalysis/berdl_fc_reference_flux")

gene_fold_changes = {}
for strain in target_strains:
    gene_fold_changes[strain] = util.load(f"ADP1BERDLFoldChangeAnalysis/berdl_fc_{strain}_vs_ADP1")

# Load model for gene-reaction rules
model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")

# Helper: parse gene_reaction_rule to extract gene IDs
def extract_gene_ids(rule):
    """Extract gene IDs from a gene_reaction_rule string."""
    if not rule or rule.strip() == "":
        return []
    # Remove 'and', 'or', parentheses, then split
    cleaned = rule.replace("(", " ").replace(")", " ")
    tokens = cleaned.split()
    gene_ids = [t for t in tokens if t.lower() not in ("and", "or") and t.strip()]
    return gene_ids

# Helper: compute geometric mean of fold changes for a set of genes
def geometric_mean_fc(gene_ids, gene_fc_dict):
    """Compute geometric mean of fold changes for genes found in the FC dict."""
    fcs = []
    for gid in gene_ids:
        if gid in gene_fc_dict:
            fc_val = gene_fc_dict[gid]
            if fc_val > 0:  # Fold changes should be positive
                fcs.append(fc_val)
    if not fcs:
        return None  # No fold change data available
    # Geometric mean = exp(mean(log(values)))
    return float(np.exp(np.mean(np.log(fcs))))

# Thresholds for near-zero flux
NEAR_ZERO = 1e-6
LARGE_FC = 100.0  # Cap for when reference is zero but fitted is not

# Build structured dict: strain -> reaction_id -> {proteomics_fc, flux_fc, fitted_flux}
three_channels = {}

for strain in target_strains:
    condition_key = full_condition_names[strain]
    condition_data = fc_results.get(condition_key, {})
    
    if condition_data.get("status") == "error":
        print(f"WARNING: {strain} has error status, skipping")
        continue
    
    fitted_fluxes = condition_data.get("fluxes", {})
    gene_fc = gene_fold_changes[strain]
    
    strain_data = {}
    
    for rxn in model.model.reactions:
        rxn_id = rxn.id
        
        # 1. Proteomics fold change: gene-level FC mapped via gene-reaction rules
        gene_ids = extract_gene_ids(rxn.gene_reaction_rule)
        prot_fc = geometric_mean_fc(gene_ids, gene_fc)
        
        # 2. Flux fold change: abs(fitted) / abs(reference)
        fitted_val = fitted_fluxes.get(rxn_id, 0.0)
        ref_val = reference_flux.get(rxn_id, 0.0)
        
        abs_fitted = abs(fitted_val)
        abs_ref = abs(ref_val)
        
        if abs_ref < NEAR_ZERO and abs_fitted < NEAR_ZERO:
            flux_fc = 1.0  # Both near zero -> no change
        elif abs_ref < NEAR_ZERO:
            flux_fc = LARGE_FC  # Reference is zero, fitted is not -> large increase
        else:
            flux_fc = abs_fitted / abs_ref
        
        # 3. Fitted flux value
        strain_data[rxn_id] = {
            "proteomics_fc": prot_fc,
            "flux_fc": round(flux_fc, 4),
            "fitted_flux": round(fitted_val, 6)
        }
    
    three_channels[strain] = strain_data
    
    # Summary stats
    n_with_prot = sum(1 for v in strain_data.values() if v["proteomics_fc"] is not None)
    n_flux_changed = sum(1 for v in strain_data.values() if abs(v["flux_fc"] - 1.0) > 0.01)
    print(f"{strain}: {len(strain_data)} reactions, {n_with_prot} with proteomics FC, {n_flux_changed} with flux FC != 1.0")

# Save three-channel data
util.save("cross_sample_three_channels", three_channels)
print(f"\nSaved three-channel data for {len(three_channels)} strains.")

2026-03-16 13:08:02,954 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-03-16 13:08:02,955 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-03-16 13:08:02,955 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-03-16 13:08:02,959 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


/Users/chenry/Dropbox/Projects/KBUtilLib/src


2026-03-16 13:08:03,346 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-03-16 13:08:03,347 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLCrossSampleAnalysis
2026-03-16 13:08:03,348 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-03-16 13:08:03,349 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


/Users/chenry/.npm-global/bin/claude


2026-03-16 13:08:03,590 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code
2026-03-16 13:08:03,601 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1BERDLCrossSampleAnalysis, loading from base datacache
2026-03-16 13:08:03,645 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1BERDLCrossSampleAnalysis, loading from base datacache
2026-03-16 13:08:03,647 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1BERDLCrossSampleAnalysis, loading from base datacache
2026-03-16 13:08:03,649 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1BERDLCrossSampleAnalysis, loading from base datacache
2026-03-16 13:08:03,650 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/

ACN2586: 1504 reactions, 1132 with proteomics FC, 545 with flux FC != 1.0
ACN2821: 1504 reactions, 1132 with proteomics FC, 550 with flux FC != 1.0
ACN3425: 1504 reactions, 1132 with proteomics FC, 549 with flux FC != 1.0
ACN3427: 1504 reactions, 1132 with proteomics FC, 546 with flux FC != 1.0
ACN3429: 1504 reactions, 1132 with proteomics FC, 560 with flux FC != 1.0
ACN3430: 1504 reactions, 1132 with proteomics FC, 550 with flux FC != 1.0

Saved three-channel data for 6 strains.


## Section C: Cross-Sample Consistency Analysis

For each reaction across all 6 strains:
1. Count strains with significant proteomics/flux fold changes
2. Classify agreement/conflict between proteomics and flux
3. Flag consistently changed and consistently conflicting reactions
4. Export summary to Excel

In [3]:
%run util.py
import pandas as pd
import numpy as np

# Load three-channel data
three_channels = util.load("cross_sample_three_channels")
strain_info = util.load("cross_sample_strains")
target_strains = strain_info["target_strains"]

# Load model for reaction names
model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")
rxn_names = {rxn.id: rxn.name for rxn in model.model.reactions}

# Thresholds for significance
FC_UP = 1.5
FC_DOWN = 1.0 / FC_UP  # 0.6667
CONSISTENCY_THRESHOLD = 4  # Number of strains that must agree

# Collect all reaction IDs across all strains
all_rxn_ids = set()
for strain in target_strains:
    if strain in three_channels:
        all_rxn_ids.update(three_channels[strain].keys())

print(f"Analyzing {len(all_rxn_ids)} reactions across {len(target_strains)} strains")
print(f"Significance thresholds: FC > {FC_UP} (up), FC < {FC_DOWN:.4f} (down)")
print(f"Consistency threshold: {CONSISTENCY_THRESHOLD}+ strains")
print("=" * 80)

# Build per-reaction summary
reaction_summary = []

for rxn_id in sorted(all_rxn_ids):
    # Track counts per reaction
    prot_up = 0
    prot_down = 0
    flux_up = 0
    flux_down = 0
    
    agree_up = 0
    agree_down = 0
    conflict_prot_up_flux_down = 0
    conflict_prot_down_flux_up = 0
    neutral = 0
    
    n_strains_with_data = 0
    
    for strain in target_strains:
        if strain not in three_channels or rxn_id not in three_channels[strain]:
            continue
        
        data = three_channels[strain][rxn_id]
        prot_fc = data["proteomics_fc"]
        flux_fc_val = data["flux_fc"]
        
        n_strains_with_data += 1
        
        # Proteomics direction
        prot_is_up = prot_fc is not None and prot_fc > FC_UP
        prot_is_down = prot_fc is not None and prot_fc < FC_DOWN
        
        # Flux direction
        flux_is_up = flux_fc_val > FC_UP
        flux_is_down = flux_fc_val < FC_DOWN
        
        if prot_is_up:
            prot_up += 1
        if prot_is_down:
            prot_down += 1
        if flux_is_up:
            flux_up += 1
        if flux_is_down:
            flux_down += 1
        
        # Agreement/Conflict classification
        if prot_is_up and flux_is_up:
            agree_up += 1
        elif prot_is_down and flux_is_down:
            agree_down += 1
        elif prot_is_up and flux_is_down:
            conflict_prot_up_flux_down += 1
        elif prot_is_down and flux_is_up:
            conflict_prot_down_flux_up += 1
        else:
            neutral += 1
    
    # Determine consistency flags
    consistently_agree_up = agree_up >= CONSISTENCY_THRESHOLD
    consistently_agree_down = agree_down >= CONSISTENCY_THRESHOLD
    consistently_changed = (
        (prot_up >= CONSISTENCY_THRESHOLD or prot_down >= CONSISTENCY_THRESHOLD) and
        (flux_up >= CONSISTENCY_THRESHOLD or flux_down >= CONSISTENCY_THRESHOLD)
    )
    consistently_conflict = (
        conflict_prot_up_flux_down >= CONSISTENCY_THRESHOLD or
        conflict_prot_down_flux_up >= CONSISTENCY_THRESHOLD
    )
    
    reaction_summary.append({
        "reaction_id": rxn_id,
        "reaction_name": rxn_names.get(rxn_id, ""),
        "n_strains": n_strains_with_data,
        "prot_up": prot_up,
        "prot_down": prot_down,
        "flux_up": flux_up,
        "flux_down": flux_down,
        "agree_up": agree_up,
        "agree_down": agree_down,
        "conflict_prot_up_flux_down": conflict_prot_up_flux_down,
        "conflict_prot_down_flux_up": conflict_prot_down_flux_up,
        "neutral": neutral,
        "consistently_changed": consistently_changed,
        "consistently_agree_up": consistently_agree_up,
        "consistently_agree_down": consistently_agree_down,
        "consistently_conflict": consistently_conflict
    })

summary_df = pd.DataFrame(reaction_summary)

# Print summary statistics
n_consist_changed = summary_df["consistently_changed"].sum()
n_consist_agree_up = summary_df["consistently_agree_up"].sum()
n_consist_agree_down = summary_df["consistently_agree_down"].sum()
n_consist_conflict = summary_df["consistently_conflict"].sum()

print(f"\nCross-Sample Consistency Summary:")
print(f"  Total reactions analyzed: {len(summary_df)}")
print(f"  Consistently changed (4+ strains, both prot & flux): {n_consist_changed}")
print(f"  Consistently agree UP (4+ strains): {n_consist_agree_up}")
print(f"  Consistently agree DOWN (4+ strains): {n_consist_agree_down}")
print(f"  Consistently conflict (4+ strains): {n_consist_conflict}")

# Show consistently agreed reactions
if n_consist_agree_up > 0:
    print(f"\n--- Reactions consistently AGREED UP ({n_consist_agree_up}) ---")
    for _, row in summary_df[summary_df["consistently_agree_up"]].iterrows():
        print(f"  {row['reaction_id']}: {row['reaction_name']} (agree_up={row['agree_up']}/{row['n_strains']})")

if n_consist_agree_down > 0:
    print(f"\n--- Reactions consistently AGREED DOWN ({n_consist_agree_down}) ---")
    for _, row in summary_df[summary_df["consistently_agree_down"]].iterrows():
        print(f"  {row['reaction_id']}: {row['reaction_name']} (agree_down={row['agree_down']}/{row['n_strains']})")

if n_consist_conflict > 0:
    print(f"\n--- Reactions consistently CONFLICTING ({n_consist_conflict}) ---")
    for _, row in summary_df[summary_df["consistently_conflict"]].iterrows():
        conflict_type = "ProtUp-FluxDown" if row["conflict_prot_up_flux_down"] >= CONSISTENCY_THRESHOLD else "ProtDown-FluxUp"
        count = max(row["conflict_prot_up_flux_down"], row["conflict_prot_down_flux_up"])
        print(f"  {row['reaction_id']}: {row['reaction_name']} ({conflict_type}, count={count}/{row['n_strains']})")

# Export to Excel
output_subdir = util.output_dir + "/ADP1BERDLCrossSampleAnalysis"
os.makedirs(output_subdir, exist_ok=True)
excel_path = output_subdir + "/cross_sample_summary.xlsx"
summary_df.to_excel(excel_path, index=False)
print(f"\nExported summary to: {excel_path}")

# Save to datacache for subsequent cells
util.save("cross_sample_summary", summary_df.to_dict(orient="list"))
print("Saved cross-sample summary to datacache.")

2026-03-16 13:08:06,468 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-03-16 13:08:06,469 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-03-16 13:08:06,470 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-03-16 13:08:06,471 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


/Users/chenry/Dropbox/Projects/KBUtilLib/src


2026-03-16 13:08:06,864 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-03-16 13:08:06,866 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLCrossSampleAnalysis
2026-03-16 13:08:06,867 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-03-16 13:08:06,867 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


/Users/chenry/.npm-global/bin/claude


2026-03-16 13:08:07,111 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code


Analyzing 1504 reactions across 6 strains
Significance thresholds: FC > 1.5 (up), FC < 0.6667 (down)
Consistency threshold: 4+ strains

Cross-Sample Consistency Summary:
  Total reactions analyzed: 1504
  Consistently changed (4+ strains, both prot & flux): 36
  Consistently agree UP (4+ strains): 5
  Consistently agree DOWN (4+ strains): 10
  Consistently conflict (4+ strains): 13

--- Reactions consistently AGREED UP (5) ---
  DgoA: DgoA (agree_up=6/6)
  rxn02782_c0: 2,5-Dihydro-5-oxofuran-2-acetate lyase (decyclizing) [c0] (agree_up=5/6)
  rxn02971_c0: 5-oxo-2,5-dihydrofuran-2-acetate delta3-delat2-isomerase [c0] (agree_up=5/6)
  rxn05582_c0: TRANS-RXNBWI-115637.ce.maizeexp.GLY_GLY [c0] (agree_up=6/6)
  rxn12386_c0: transport of dodecanoate [extraorganism;cytosol](secondary symport) (agree_up=4/6)

--- Reactions consistently AGREED DOWN (10) ---
  rxn00060_c0: porphobilinogen:(4-[2-carboxyethyl]-3-[carboxymethyl]pyrrol-2-yl)methyltransferase (hydrolysing) [c0] (agree_down=4/6)
  rxn

## Section D: Generate Enhanced Escher Maps

Generate one map per strain (6 maps) with reaction badges showing:
- Proteomics FC
- Flux FC

Plus one reference flux map (no badges).

In [4]:
%run util.py

# Load all required data
strain_info = util.load("cross_sample_strains")
target_strains = strain_info["target_strains"]
full_condition_names = strain_info["full_condition_names"]

three_channels = util.load("cross_sample_three_channels")
fc_results = util.load("ADP1BERDLFoldChangeAnalysis/ADP1BERDLFoldChangeAnalysis")
reference_flux = util.load("ADP1BERDLFoldChangeAnalysis/berdl_fc_reference_flux")

# Load model
model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")

# Output directory
output_subdir = util.output_dir + "/ADP1BERDLCrossSampleAnalysis"
os.makedirs(output_subdir, exist_ok=True)

generated_maps = {}

# Generate reference flux map (no badges)
print("Generating reference flux map...")
ref_path = util.create_map_html2(
    model=model.model,
    flux=reference_flux,
    map="full",
    output_path=output_subdir + "/map_reference.html",
)
generated_maps["reference"] = ref_path
print(f"  Reference map: {ref_path}")

# Generate one map per strain with reaction badges
for strain in target_strains:
    print(f"\nGenerating map for {strain}...")
    
    condition_key = full_condition_names[strain]
    condition_data = fc_results.get(condition_key, {})
    
    if condition_data.get("status") == "error":
        print(f"  SKIPPED - Error in analysis for {strain}")
        continue
    
    fitted_flux = condition_data.get("fluxes", {})
    strain_channels = three_channels.get(strain, {})
    
    # Build proteomics FC dict and flux FC dict for badges
    proteomics_fc_dict = {}
    flux_fc_dict = {}
    
    for rxn_id, channels in strain_channels.items():
        if channels["proteomics_fc"] is not None:
            proteomics_fc_dict[rxn_id] = round(channels["proteomics_fc"], 3)
        flux_fc_dict[rxn_id] = round(channels["flux_fc"], 3)
    
    # Generate map with reaction badges
    map_path = util.create_map_html2(
        model=model.model,
        flux=fitted_flux,
        map="full",
        output_path=output_subdir + f"/map_{strain}.html",
        reaction_badges=[
            {"label": "Prot FC", "data": proteomics_fc_dict},
            {"label": "Flux FC", "data": flux_fc_dict},
        ],
        badge_saturation=3.0,
        badge_size=1.0,
    )
    generated_maps[strain] = map_path
    print(f"  Map saved: {map_path}")
    print(f"  Badges: {len(proteomics_fc_dict)} proteomics FC, {len(flux_fc_dict)} flux FC")

# Save generated map paths
util.save("cross_sample_map_paths", generated_maps)

print(f"\n{'=' * 80}")
print(f"Generated {len(generated_maps)} maps total.")

2026-03-16 13:08:11,032 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-03-16 13:08:11,033 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-03-16 13:08:11,033 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-03-16 13:08:11,035 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


/Users/chenry/Dropbox/Projects/KBUtilLib/src


2026-03-16 13:08:11,430 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-03-16 13:08:11,432 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLCrossSampleAnalysis
2026-03-16 13:08:11,432 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-03-16 13:08:11,433 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


/Users/chenry/.npm-global/bin/claude


2026-03-16 13:08:11,677 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code
2026-03-16 13:08:11,688 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1BERDLCrossSampleAnalysis, loading from base datacache
2026-03-16 13:08:11,734 - __main__.NotebookUtil - INFO - File not found in /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1BERDLCrossSampleAnalysis, loading from base datacache
2026-03-16 13:08:12,107 - __main__.NotebookUtil - INFO - Loaded map 'full' from local index: /Users/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-03-16 13:08:12,112 - __main__.NotebookUtil - INFO - Updated names for 759 reactions in map


...

  Map saved: /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/nboutput/ADP1BERDLCrossSampleAnalysis/ADP1BERDLCrossSampleAnalysis/map_ACN3427.html
  Badges: 1132 proteomics FC, 1504 flux FC

Generating map for ACN3429...


2026-03-16 13:08:19,732 - __main__.NotebookUtil - INFO - Injected numerical badge overlays: 2 channels, 2636 total badge entries
2026-03-16 13:08:19,751 - __main__.NotebookUtil - INFO - Loaded map 'full' from local index: /Users/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-03-16 13:08:19,753 - __main__.NotebookUtil - INFO - Updated names for 759 reactions in map


  Map saved: /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/nboutput/ADP1BERDLCrossSampleAnalysis/ADP1BERDLCrossSampleAnalysis/map_ACN3429.html
  Badges: 1132 proteomics FC, 1504 flux FC

Generating map for ACN3430...


2026-03-16 13:08:20,970 - __main__.NotebookUtil - INFO - Injected numerical badge overlays: 2 channels, 2636 total badge entries


  Map saved: /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/nboutput/ADP1BERDLCrossSampleAnalysis/ADP1BERDLCrossSampleAnalysis/map_ACN3430.html
  Badges: 1132 proteomics FC, 1504 flux FC

Generated 7 maps total.


## Section E: HTML Index Page

Create an index page linking all generated maps with a legend explaining the data channels.

In [5]:
%run util.py

# Load map paths and strain info
generated_maps = util.load("cross_sample_map_paths")
strain_info = util.load("cross_sample_strains")
target_strains = strain_info["target_strains"]

output_subdir = util.output_dir + "/ADP1BERDLCrossSampleAnalysis"
os.makedirs(output_subdir, exist_ok=True)

# Build strain map links
strain_links = ""
for strain in target_strains:
    if strain in generated_maps:
        filename = f"map_{strain}.html"
        strain_links += f'        <li><a href="{filename}">{strain} &mdash; Fitted Flux with Badges</a></li>\n'

# Reference link
ref_link = ""
if "reference" in generated_maps:
    ref_link = '<li><a href="map_reference.html">ADP1 Reference Flux (BERDL fitness-constrained)</a></li>'

html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>ADP1 BERDL Fold Change Analysis &mdash; Proteomics + Flux Maps</title>
    <style>
        body {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
            max-width: 800px;
            margin: 40px auto;
            padding: 0 20px;
            color: #333;
            line-height: 1.6;
        }}
        h1 {{
            color: #1a1a2e;
            border-bottom: 2px solid #16213e;
            padding-bottom: 10px;
        }}
        h2 {{
            color: #16213e;
            margin-top: 30px;
        }}
        ul {{
            list-style-type: none;
            padding: 0;
        }}
        li {{
            margin: 8px 0;
            padding: 8px 12px;
            background: #f8f9fa;
            border-radius: 4px;
            border-left: 3px solid #16213e;
        }}
        a {{
            color: #0f3460;
            text-decoration: none;
            font-weight: 500;
        }}
        a:hover {{
            text-decoration: underline;
        }}
        .legend {{
            background: #f0f4f8;
            border: 1px solid #d1d9e6;
            border-radius: 6px;
            padding: 16px 20px;
            margin: 20px 0;
        }}
        .legend h3 {{
            margin-top: 0;
            color: #16213e;
        }}
        .channel {{
            margin: 8px 0;
        }}
        .channel strong {{
            display: inline-block;
            width: 120px;
        }}
        .footer {{
            margin-top: 40px;
            padding-top: 10px;
            border-top: 1px solid #eee;
            font-size: 0.85em;
            color: #888;
        }}
    </style>
</head>
<body>
    <h1>ADP1 BERDL Fold Change Analysis</h1>
    <p>Proteomics + Flux Maps for 6 mutant strains vs ADP1 wild-type reference.</p>

    <div class="legend">
        <h3>Data Channels Legend</h3>
        <p>Each strain map shows the fitted flux distribution with two reaction badges:</p>
        <div class="channel"><strong>Flux (arrows):</strong> Fitted flux magnitude and direction from fold-change-constrained FBA.</div>
        <div class="channel"><strong>Prot FC (badge):</strong> Proteomics fold change &mdash; gene-level fold change mapped to reactions via gene-reaction rules (geometric mean for multi-gene reactions). Values &gt;1 indicate upregulation vs ADP1; &lt;1 indicate downregulation.</div>
        <div class="channel"><strong>Flux FC (badge):</strong> Flux fold change &mdash; abs(fitted flux) / abs(reference flux). Values &gt;1 indicate increased flux; &lt;1 indicate decreased flux relative to the ADP1 BERDL reference.</div>
        <p>Badge colors: <span style="color:#c0392b;">red</span> = upregulated/increased, <span style="color:#2980b9;">blue</span> = downregulated/decreased, saturating at {3.0}x.</p>
    </div>

    <h2>Strain Maps</h2>
    <ul>
{strain_links}    </ul>

    <h2>Reference Map</h2>
    <ul>
        {ref_link}
    </ul>

    <h2>Supplementary Data</h2>
    <ul>
        <li><a href="cross_sample_summary.xlsx">Cross-Sample Consistency Summary (Excel)</a></li>
    </ul>

    <div class="footer">
        Generated by ADP1BERDLCrossSampleAnalysis notebook.
    </div>
</body>
</html>
"""

index_path = output_subdir + "/index.html"
with open(index_path, "w") as f:
    f.write(html_content)

print(f"Index page saved to: {index_path}")

# Display link
from IPython.display import HTML, display
display(HTML(f'<a href="{index_path}" target="_blank">Open Index Page</a>'))

2026-03-16 13:09:29,751 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-03-16 13:09:29,751 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-03-16 13:09:29,752 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-03-16 13:09:29,754 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


/Users/chenry/Dropbox/Projects/KBUtilLib/src


2026-03-16 13:09:30,167 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-03-16 13:09:30,169 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLCrossSampleAnalysis
2026-03-16 13:09:30,170 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-03-16 13:09:30,170 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


/Users/chenry/.npm-global/bin/claude


2026-03-16 13:09:30,420 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code


Index page saved to: /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/nboutput/ADP1BERDLCrossSampleAnalysis/ADP1BERDLCrossSampleAnalysis/index.html
